In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [3]:
df = pd.read_csv('data/NB_training-set.csv')

In [5]:
df.isnull().sum()



id                   0
dur                  0
proto                0
service              0
state                0
spkts                0
dpkts                0
sbytes               0
dbytes               0
rate                 0
sttl                 0
dttl                 0
sload                0
dload                0
sloss                0
dloss                0
sinpkt               0
dinpkt               0
sjit                 0
djit                 0
swin                 0
stcpb                0
dtcpb                0
dwin                 0
tcprtt               0
synack               0
ackdat               0
smean                0
dmean                0
trans_depth          0
response_body_len    0
ct_srv_src           0
ct_state_ttl         0
ct_dst_ltm           0
ct_src_dport_ltm     0
ct_dst_sport_ltm     0
ct_dst_src_ltm       0
is_ftp_login         0
ct_ftp_cmd           0
ct_flw_http_mthd     0
ct_src_ltm           0
ct_srv_dst           0
is_sm_ips_ports      0
attack_cat 

In [6]:
columns_to_drop = ['id', 'attack_cat']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [7]:
categorical_cols = ['proto', 'service', 'state']
label_encoders = {}

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).fillna('missing')
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

In [11]:
X_train_full = df.drop(columns=['label'])
y_train_full = df['label']

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=0.15, 
    random_state=42, 
    stratify=y_train_full
)

In [13]:
# Calculate scale_pos_weight if data is imbalanced (count of negative / count of positive)
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
scale_weight = neg_count / pos_count if pos_count > 0 else 1.0

model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_weight,
    random_state=42,
    early_stopping_rounds=50, # Stops if validation score doesn't improve for 50 rounds
    eval_metric="logloss"
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-logloss:0.65402
[100]	validation_0-logloss:0.10077
[200]	validation_0-logloss:0.07476
[300]	validation_0-logloss:0.06669
[400]	validation_0-logloss:0.06293
[500]	validation_0-logloss:0.06073
[600]	validation_0-logloss:0.05864
[700]	validation_0-logloss:0.05721
[800]	validation_0-logloss:0.05610
[900]	validation_0-logloss:0.05574
[999]	validation_0-logloss:0.05460


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [14]:
y_val_pred = model.predict(X_val)
print("\nValidation Internal Slice Report:")
print(classification_report(y_val, y_val_pred))


Validation Internal Slice Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      5550
           1       0.99      0.98      0.98      6800

    accuracy                           0.98     12350
   macro avg       0.98      0.98      0.98     12350
weighted avg       0.98      0.98      0.98     12350



In [15]:
print("Saving artifacts...")

joblib.dump(label_encoders, 'label_encoders.pkl')
joblib.dump(X_train_full.columns.tolist(), 'feature_columns.pkl')
print("Model preparation complete!")

Saving artifacts...
Model preparation complete!


In [17]:
df_test = pd.read_csv('data/NB_testing-set.csv')


df_test = df_test.drop(columns=[col for col in ['id', 'attack_cat'] if col in df_test.columns])

# Load Encoders and Feature order
label_encoders = joblib.load('label_encoders.pkl')
expected_features = joblib.load('feature_columns.pkl')

In [18]:
for col, le in label_encoders.items():
    if col in df_test.columns:
        df_test[col] = df_test[col].astype(str).fillna('missing')
        # Handle unseen categories gracefully by mapping them to an 'unknown' placeholder if necessary
        df_test[col] = df_test[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

X_test = df_test[expected_features]
y_test = df_test['label']

In [19]:
y_test_pred = model.predict(X_test)

print("\nFinal Test Dataset Performance:")
print(classification_report(y_test, y_test_pred))


Final Test Dataset Performance:
              precision    recall  f1-score   support

           0       0.77      0.98      0.86     56000
           1       0.99      0.86      0.92    119341

    accuracy                           0.90    175341
   macro avg       0.88      0.92      0.89    175341
weighted avg       0.92      0.90      0.90    175341



In [20]:
joblib.dump(model, 'xgboost_network_model.pkl')
print('model has been saved')

model has been saved
